# JEM MNIST Pipeline — Run, Analyze, Plot

Companion to `mnist.ipynb` for the real-degree-of-freedom-matched MLP/JEM baseline (`144 → 550 → 480 → 10`, 349,040 real parameters). It follows the same ordered workflow and produces the same main figure families, replacing MPS likelihood/Gibbs operations with JEM score-gradient/SGLD operations.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import data_root
from baselines.jem.plots import (
    plot_alpha_curves, plot_defense_comparison,
    plot_detection_thresholds, plot_purification_radii,
)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {data_root()}")

---
## §0. HPO

Run the α=0 HPO first, then its seed sweep. The selected α=0 checkpoint warm-starts every α>0 HPO.

In [ ]:
print("python -m baselines.jem.train --multirun +experiment=hpo/a0 device=cuda:0")
print("# HPO writes the selected alpha=0 HPs automatically; then run:")
print("python -m baselines.jem.train --multirun +experiment=seed_sweep/a0 device=cuda:0")
A0_MODEL = "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a0_DDMM/2/models/model.pt"

---
## §1. Generative HPO, seed sweeps and AT baseline

For α>0, Optuna minimizes validation mixed CD using the fixed stronger validation sampler; training-SGLD hyperparameters remain candidates. At the end of each HPO, the selected values are written automatically into the matching seed-sweep YAML.

In [ ]:
for name, alpha in [("a001", 0.01), ("a01", 0.1), ("a02", 0.2), ("a05", 0.5), ("a1", 1.0)]:
    print("python -m baselines.jem.train --multirun +experiment=hpo/pretrained "
          f"run_name=pretrained_{name} trainer.alpha={alpha} model_path={A0_MODEL} device=cuda:0")
    print(f"# Then: python -m baselines.jem.train --multirun +experiment=seed_sweep/{name} model_path={A0_MODEL} device=cuda:0")
print(f"python -m baselines.jem.train --multirun +experiment=hpo/at model_path={A0_MODEL} device=cuda:0")
print(f"python -m baselines.jem.train --multirun +experiment=seed_sweep/at model_path={A0_MODEL} device=cuda:0")

---
## §2. Post-hoc analysis

Fill the dated output directories. `baselines.jem.sweep` generates `evaluation_data.csv`, a numeric summary CSV and the compact MPS-style `evaluation_summary.txt`.

In [ ]:
SWEEP_DIRS = {
    "a0":   "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a0_DDMM",
    "a001": "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a001_DDMM",
    "a01":  "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a01_DDMM",
    "a02":  "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a02_DDMM",
    "a05":  "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a05_DDMM",
    "a1":   "outputs/baselines/jem/mnist_full_r12/nat/mlp_h550x480/seed_sweep/a1_DDMM",
    "at":   "outputs/baselines/jem/mnist_full_r12/at/mlp_h550x480/seed_sweep/at_DDMM",
}
for path in SWEEP_DIRS.values():
    print(f"python -m baselines.jem.sweep {path} --device cuda:0")

In [ ]:
ANALYSIS_CSV = {
    key: data_root() / "analysis/outputs" / Path(path).relative_to("outputs") / "evaluation_data.csv"
    for key, path in SWEEP_DIRS.items()
}
for key, path in ANALYSIS_CSV.items():
    print(key, path, "OK" if path.exists() else "MISSING")

---
## §3. Sampling

Final generation uses long class-conditional chains and is intentionally distinct from the short training sampler.

In [ ]:
import pandas as pd
sample_df = pd.read_csv(ANALYSIS_CSV["a001"])
SAMPLE_RUN = sample_df.loc[sample_df["acc"].idxmax(), "run_path"]
print(f"python -m baselines.jem.generate {SAMPLE_RUN} --steps 1000 --per-class 8 --device cuda:0")

---
## §4. Alpha curves

Same clean/robust/purified accuracy figure as the MPS notebook. The loss panel labels the JEM quantity correctly as a CD surrogate rather than exact NLL.

In [ ]:
alpha_csv = {
    0.0: ANALYSIS_CSV["a0"], 0.01: ANALYSIS_CSV["a001"],
    0.1: ANALYSIS_CSV["a01"], 0.2: ANALYSIS_CSV["a02"],
    0.5: ANALYSIS_CSV["a05"], 1.0: ANALYSIS_CSV["a1"],
}
plot_alpha_curves(alpha_csv, PROJECT_ROOT / "figures/jem_mnist/alpha", epsilon=0.3, radius=0.2)

---
## §5. Purification radii and defense comparison

Gradient ascent is the likelihood-purification analogue; projected SGLD is the JEM-native stochastic defense.

In [ ]:
headline = {
    r"$\alpha=0$": ANALYSIS_CSV["a0"],
    r"$\alpha=0.01$": ANALYSIS_CSV["a001"],
    r"$\alpha=0.5$": ANALYSIS_CSV["a05"],
    "MLP-AT": ANALYSIS_CSV["at"],
}
plot_purification_radii(headline, PROJECT_ROOT / "figures/jem_mnist/purify_radius/gradient.png")
plot_purification_radii(headline, PROJECT_ROOT / "figures/jem_mnist/purify_radius/sgld.png", method="sgld")
plot_defense_comparison(headline, PROJECT_ROOT / "figures/jem_mnist/robustness", radius=0.2)

---
## §6. Likelihood/energy detection curves

Solid lines show accuracy among accepted inputs; dashed lines show detection rate, matching the MPS MNIST notebook.

In [ ]:
detection_models = {
    r"$\alpha=0$": ANALYSIS_CSV["a0"],
    r"$\alpha=0.01$": ANALYSIS_CSV["a001"],
    "MLP-AT": ANALYSIS_CSV["at"],
}
plot_detection_thresholds(detection_models, PROJECT_ROOT / "figures/jem_mnist/detection")